# Track 4b: build our own NP-signal dataset and run the offline benchmark (Colab)

Plan: `specs/track4_priors_probes/np_signals_plan.md`. The gates were pre-registered in
`specs/track4_priors_probes/gate_report.md` (Track 4b register, NB.*) **before** this run.
**Generated by `build/mk_np_signals.py`; do not hand-edit.** The core functions live in
`build/np_signals_core.py` (tested locally by `build/test_np_signals_core.py`).

Runtime: **CPU, High-RAM**, about 60–90 min. Most of that is downloading and streaming SID-Map; the benchmark is
multiprocessing.

| step | what |
|---|---|
| 1 | inputs: PubChem tier, Gluzdov popularity v2 and NP knowledge table from Drive (tier and popularity arrays copied to local disk); SID-Map, Source-Names, source table downloaded from PubChem |
| 2 | `vendor_only` per CID: every live SID comes from a "Chemical Vendors" source and PMID = 0 → tier-aligned and pool-aligned arrays; NB.V0 |
| 3 | labels: NP-table superclass by IK14; a LOTUS fingerprint bank for kNN (Tanimoto ≥ 0.7); training-structure labels |
| 4 | cohorts T (training structures) and L (NP table, no training spectrum), 2,000 each |
| 5 | benchmark: the truth among its same-formula tier isomers, `pop` vs `pop + μ·signal` → NB.0, NB.V, NB.N, NB.C0, NB.C |

Outputs go to Drive `casmi_2026/_np_signals/`. Upload that folder as the private Kaggle dataset `casmi26-np-signals`.
Paste the final printout back into the chat.

In [ ]:
import os, sys, json, time, subprocess, gzip, urllib.request
import multiprocessing as mp
from pathlib import Path
import numpy as np
import pandas as pd

from google.colab import drive
drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive/casmi_2026")
COMP = ROOT / "enveda-CASMI26-molecule-id-mass-spectra"
TRAIN = COMP / "train.parquet"
OUT = ROOT / "_np_signals"
WORK = Path("/content/np_signals")
DL = WORK / "dl"
for p in (OUT, WORK, DL):
    p.mkdir(parents=True, exist_ok=True)
# an offline wheel is only usable if it matches this Python (the Drive wheel is cp312, built for Kaggle;
# Colab may run a newer Python) -- otherwise pip installs the same pinned version from PyPI
PY_TAG = f"cp{sys.version_info.major}{sys.version_info.minor}"
RDKIT_WHL = next(iter(sorted((ROOT / "casmi26-offline-rdkit-2026033").glob(f"rdkit-*-{PY_TAG}-*.whl"))), None)
RDKIT_PIN = "2026.03.3"

# the three Kaggle datasets, already on Drive
TIER_DRIVE = ROOT / "casmi26-pubchem-tier"
POP_DRIVE = ROOT / "casmi26-pubchem-popularity-prior"          # needs Version 2 (pc_cid, pc_ik14, pc_np)
NPT_DRIVE = ROOT / "casmi26-natural-product-knowledge-table"
COPY_LOCAL = True   # the benchmark reads the tier randomly; local disk is far faster than the Drive FUSE mount
FTP = "https://ftp.ncbi.nlm.nih.gov/pubchem/Substance/Extras/"
SRCTAB_URL = "https://pubchem.ncbi.nlm.nih.gov/rest/pug/sourcetable/substance/CSV"

# pre-registered (gate_report.md, Track 4b register): do not edit after the run starts
N_COHORT, SEED, PPM_WIN = 2000, 0, 2.0
GATE_L, GATE_T = 0.010, -0.005
GATE_V0 = 0.01
GATE_C0_ANALOG, GATE_C0_MULTI = 0.50, 0.20

NP_CORE_SRC = '"""Track 4b NP signals: the core functions shared by the Colab build/benchmark and the local tests.\n\nEmbedded verbatim into casmi_np_signals_colab.ipynb (by build/mk_np_signals.py) and written back to a file,\nso the multiprocessing workers can import it. Plan: specs/track4_priors_probes/np_signals_plan.md.\n\nThree signals, each meant as one additive term on top of the borrowed popularity prior\npop = log1p(SID) + log1p(PMID):\n  vendor_only  1 if every live SID of the CID was deposited by a "Chemical Vendors" source and PMID == 0\n  np_like      Ertl NP-likeness (RDKit Contrib NP_Score), z-scored within each candidate window\n  class_cons   superclass agreement between a candidate and the molecule\'s nearest library structures,\n               similarity-weighted. A candidate with no label scores 0, never a penalty.\n"""\nfrom __future__ import annotations\n\nimport os\nimport re\nimport sys\n\nimport numpy as np\n\n# ---------------------------------------------------------------- metrics\n\ndef expected_rank_metrics(scores, truth_idx, k=25, hits=(1, 5, 25)):\n    """Expected reciprocal rank @k and hit@n of scores[truth_idx] under uniformly random tie-breaking.\n\n    Ties matter here: thousands of vendor decoys share pop = log1p(1) and, under a pessimistic or optimistic\n    tie rule, the ordering of equal scores would decide the metric on its own.\n    Returns (rr, {n: hit}).\n    """\n    s = np.round(np.asarray(scores, np.float64), 9)\n    t = s[truth_idx]\n    g = int((s > t).sum())\n    e = int((s == t).sum())               # includes the truth itself\n    ranks = np.arange(g + 1, g + e + 1)\n    rr = float(np.where(ranks <= k, 1.0 / ranks, 0.0).mean())\n    hit = {n: float(np.clip(n - g, 0, e) / e) for n in hits}\n    return rr, hit\n\n\n# ---------------------------------------------------------------- vendor_only\n\ndef vendor_source_ids(source_table, source_names=None):\n    """Return the set of source names/ids whose PubChem source category contains "vendor".\n\n    source_table: DataFrame from PUG REST sourcetable/substance/CSV; must have a name column and a category\n    column (case-insensitive match on \'categ\'). Raises if no category column exists, because a silent fallback\n    would make the vendor flag mean something else.\n    source_names: optional DataFrame (id, display_name) from FTP Substance/Extras/Source-Names. SID-Map\n    stores the depositor\'s source *id*, which for newer depositors is numeric, so ids are added for every\n    vendor display name.\n    """\n    cols = {c.lower(): c for c in source_table.columns}\n    cat = next((cols[c] for c in cols if \'categ\' in c), None)\n    if cat is None:\n        raise KeyError(f\'no category column in the source table: {list(source_table.columns)}\')\n    name = next((cols[c] for c in cols if c in (\'source name\', \'name\', \'sourcename\', \'source\')), None)\n    if name is None:\n        name = next(cols[c] for c in cols if \'name\' in c)\n    isv = source_table[cat].astype(str).str.contains(\'vendor\', case=False, na=False)\n    vend = set(source_table.loc[isv, name].astype(str).str.strip())\n    if source_names is not None:\n        sid_col, disp_col = source_names.columns[:2]\n        m = source_names[disp_col].astype(str).str.strip().isin(vend) | source_names[sid_col].astype(str).str.strip().isin(vend)\n        vend |= set(source_names.loc[m, sid_col].astype(str).str.strip())\n    return vend\n\n\ndef accumulate_sid_map(src, cid, vendor_set, nonvendor, anysid):\n    """One SID-Map chunk: src (array of source ids, str), cid (float/int array, NaN = no standardized CID).\n    Sets anysid[cid] for every SID and nonvendor[cid] for SIDs whose source is not a vendor. In place."""\n    cid = np.asarray(cid, np.float64)\n    ok = np.isfinite(cid) & (cid > 0) & (cid < len(nonvendor))\n    if not ok.any():\n        return 0\n    import pandas as pd\n    c = cid[ok].astype(np.int64)\n    isv = pd.Series(np.asarray(src, object)[ok]).astype(str).str.strip().isin(vendor_set).values\n    anysid[c] = True\n    nonvendor[c[~isv]] = True\n    return int(ok.sum())\n\n\ndef vendor_only_flags(cids, lpmid, nonvendor, anysid):\n    """Per-row vendor_only for an array of representative CIDs and log1p(PMID) values."""\n    cids = np.asarray(cids, np.int64)\n    ok = (cids > 0) & (cids < len(nonvendor))\n    c = np.where(ok, cids, 0)\n    no_pmid = np.asarray(lpmid, np.float32) <= 0\n    return (ok & anysid[c] & ~nonvendor[c] & no_pmid).astype(np.uint8)\n\n\n# ---------------------------------------------------------------- NP-likeness\n\n_NP = {}\n\n\ndef _np_model():\n    if \'m\' not in _NP:\n        from rdkit.Chem import RDConfig\n        p = os.path.join(RDConfig.RDContribDir, \'NP_Score\')\n        if p not in sys.path:\n            sys.path.append(p)\n        import npscorer\n        _NP[\'f\'] = npscorer\n        _NP[\'m\'] = npscorer.readNPModel()\n    return _NP[\'f\'], _NP[\'m\']\n\n\ndef np_scores(smiles):\n    """Ertl NP-likeness per SMILES; NaN where RDKit cannot parse."""\n    from rdkit import Chem, RDLogger\n    RDLogger.DisableLog(\'rdApp.*\')\n    f, m = _np_model()\n    out = np.full(len(smiles), np.nan, np.float32)\n    for i, s in enumerate(smiles):\n        mol = Chem.MolFromSmiles(s)\n        if mol is not None:\n            try:\n                out[i] = f.scoreMol(mol, m)\n            except Exception:\n                pass\n    return out\n\n\ndef zscore_window(x):\n    """z-score within one candidate window; NaN -> 0 (neutral), constant window -> all 0."""\n    x = np.asarray(x, np.float64)\n    ok = np.isfinite(x)\n    out = np.zeros(len(x))\n    if ok.sum() >= 2:\n        sd = x[ok].std()\n        if sd > 0:\n            out[ok] = (x[ok] - x[ok].mean()) / sd\n    return out\n\n\n# ---------------------------------------------------------------- class consistency\n\ndef morgan_fp(smiles, nbits=2048):\n    from rdkit import Chem, RDLogger\n    from rdkit.Chem import rdFingerprintGenerator\n    RDLogger.DisableLog(\'rdApp.*\')\n    gen = _NP.setdefault(\'gen\', rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=nbits))\n    m = Chem.MolFromSmiles(smiles)\n    return None if m is None else gen.GetFingerprint(m)\n\n\ndef knn_label(fp, bank_fps, bank_labels, thr=0.7):\n    """Label of the nearest bank structure if its Tanimoto >= thr, else None. Returns (label, sim)."""\n    if fp is None or not len(bank_fps):\n        return None, 0.0\n    from rdkit import DataStructs\n    sims = np.asarray(DataStructs.BulkTanimotoSimilarity(fp, bank_fps))\n    j = int(sims.argmax())\n    return (bank_labels[j], float(sims[j])) if sims[j] >= thr else (None, float(sims[j]))\n\n\ndef nearest_analogs(fp, bank_fps, k=5, exclude=None):\n    """Top-k bank indices by Tanimoto (excluding indices in `exclude`). Returns (idx, sims)."""\n    from rdkit import DataStructs\n    sims = np.asarray(DataStructs.BulkTanimotoSimilarity(fp, bank_fps))\n    if exclude is not None and len(exclude):\n        sims[np.asarray(list(exclude), np.int64)] = -1.0\n    idx = np.argsort(-sims)[:k]\n    idx = idx[sims[idx] > 0]\n    return idx, sims[idx]\n\n\ndef class_consistency(cand_labels, analog_labels, analog_sims):\n    """s_c = sum_k sim_k [label_c == label_k] / sum_k sim_k over labelled analogs. Unlabelled candidate -> 0;\n    no labelled analog -> all 0 (the signal abstains instead of guessing)."""\n    pairs = [(l, w) for l, w in zip(analog_labels, analog_sims) if l is not None and w > 0]\n    out = np.zeros(len(cand_labels))\n    if not pairs:\n        return out\n    tot = sum(w for _, w in pairs)\n    wt = {}\n    for l, w in pairs:\n        wt[l] = wt.get(l, 0.0) + w / tot\n    for i, l in enumerate(cand_labels):\n        if l is not None:\n            out[i] = wt.get(l, 0.0)\n    return out\n\n\n# ---------------------------------------------------------------- PubChem-tier windows\n\n_FORMULA_RE = re.compile(r\'([A-Z][a-z]?)(\\d*)\')\n\n\ndef canon_formula(f):\n    """Hill-order-insensitive formula key (\'C6H12O6\' == \'H12C6O6\'); charges dropped."""\n    f = re.sub(r\'[+-]\\d*$\', \'\', str(f))\n    d = {}\n    for el, n in _FORMULA_RE.findall(f):\n        d[el] = d.get(el, 0) + (int(n) if n else 1)\n    return \'\'.join(f\'{el}{d[el]}\' for el in sorted(d))\n\n\ndef tier_slice(mass, off, buf, target, ppm):\n    """Row range [a, b) and SMILES of the tier rows within ppm of a neutral mass."""\n    tol = target * ppm * 1e-6\n    a = int(np.searchsorted(mass, target - tol, \'left\'))\n    b = int(np.searchsorted(mass, target + tol, \'right\'))\n    if b <= a:\n        return a, b, []\n    o = np.asarray(off[a:b + 1], np.int64)\n    raw = bytes(buf[o[0]:o[-1]])\n    base = o[0]\n    return a, b, [raw[o[i] - base:o[i + 1] - base].decode(\'ascii\') for i in range(b - a)]\n\n\ndef same_formula(smiles, formula):\n    """Mask of SMILES whose molecular formula equals `formula` (canonicalised)."""\n    from rdkit import Chem, RDLogger\n    from rdkit.Chem.rdMolDescriptors import CalcMolFormula\n    RDLogger.DisableLog(\'rdApp.*\')\n    want = canon_formula(formula)\n    ok = np.zeros(len(smiles), bool)\n    for i, s in enumerate(smiles):\n        m = Chem.MolFromSmiles(s)\n        if m is not None and canon_formula(CalcMolFormula(m)) == want:\n            ok[i] = True\n    return ok\n\n\ndef dedupe_first(keys):\n    """Indices of the first occurrence of each key, in order."""\n    seen, keep = set(), []\n    for i, k in enumerate(keys):\n        if k not in seen:\n            seen.add(k)\n            keep.append(i)\n    return np.asarray(keep, np.int64)\n\n\n# ---------------------------------------------------------------- offline benchmark (one query)\n\nG = {}                 # filled by the notebook before the worker Pool forks (mmaps, banks, label maps)\nMUS = (0.25, 0.5, 1.0)\nKNN_TOP = 150          # candidates (by pop) that get a kNN label when the NP table has none\nKNN_THR = 0.7\nK_ANALOG = 5\n\n\ndef _key(k):\n    return k.decode(\'ascii\', \'ignore\') if isinstance(k, (bytes, np.bytes_)) else str(k)\n\n\ndef eval_query(task):\n    """task = (qid, ik14, formula, mono_mass, smiles, cohort). Ranks the truth among its same-formula PubChem-tier\n    isomers (deduplicated by IK14, keeping the most popular row) under pop and pop + mu * signal.\n\n    G keys: mass, off, buf (tier), ik14, lsid, lpmid, vendor (row-aligned), ppm, label_of (ik14 -> superclass),\n    lotus_fps, lotus_labels, train_fps, train_labels, train_rows_of (ik14 -> list of bank indices).\n    """\n    qid, ik14, formula, mono_mass, smiles, cohort = task\n    out = dict(qid=qid, cohort=cohort, found=False, n=0)\n    a, b, smis = tier_slice(G[\'mass\'], G[\'off\'], G[\'buf\'], float(mono_mass), G[\'ppm\'])\n    if not smis:\n        return out\n    sf = np.where(same_formula(smis, formula))[0]\n    if not len(sf):\n        return out\n    rows = a + sf\n    pop = np.asarray(G[\'lsid\'][rows], np.float32) + np.asarray(G[\'lpmid\'][rows], np.float32)\n    keys = np.array([_key(k) for k in G[\'ik14\'][rows]], object)\n    o = np.argsort(-pop, kind=\'stable\')\n    keep = o[dedupe_first(keys[o])]\n    keep = keep[keys[keep] != \'\']\n    rows, pop, keys = rows[keep], pop[keep], keys[keep]\n    excl = G.get(\'exclude_keys\') if cohort in G.get(\'exclude_cohorts\', ()) else None\n    if excl:\n        # mirror supernext\'s PubChem channel: structures already in the candidate pool are never listed there\n        m = np.array([(k == ik14) or (k not in excl) for k in keys], bool)\n        rows, pop, keys = rows[m], pop[m], keys[m]\n    t = np.where(keys == ik14)[0]\n    out[\'n\'] = int(len(rows))\n    if not len(t):\n        return out\n    t = int(t[0])\n    out[\'found\'] = True\n    sm = [smis[i - a] for i in rows]\n    vendor = np.asarray(G[\'vendor\'][rows], np.float64)\n    npz = zscore_window(np_scores(sm))\n\n    # class consistency: candidate labels (table, else kNN for the KNN_TOP most popular), analogs from the library\n    lab = [G[\'label_of\'].get(k) for k in keys]\n    for i in np.argsort(-pop, kind=\'stable\')[:KNN_TOP]:\n        if lab[i] is None:\n            lab[i], _ = knn_label(morgan_fp(sm[i]), G[\'lotus_fps\'], G[\'lotus_labels\'], KNN_THR)\n    qfp = morgan_fp(smiles)\n    if qfp is not None and len(G[\'train_fps\']):\n        ai, asim = nearest_analogs(qfp, G[\'train_fps\'], K_ANALOG, exclude=G[\'train_rows_of\'].get(ik14, ()))\n        alab = [G[\'train_labels\'][j] for j in ai]\n    else:\n        alab, asim = [], []\n    cc = class_consistency(lab, alab, asim)\n    mem = np.array([l is not None for l in lab], np.float64)   # membership-only ablation: "has an NP label"\n    top_lab = {lab[i] for i in np.argsort(-pop, kind=\'stable\')[:KNN_TOP] if lab[i] is not None}\n    out.update(has_lab_analog=any(l is not None for l in alab), multi_class_window=len(top_lab) >= 2,\n               truth_vendor=int(vendor[t]), truth_lab=lab[t] is not None, cc_truth=float(cc[t]),\n               vendor_rate=float(vendor.mean()))\n\n    variants = {\'pop\': pop}\n    for mu in MUS:\n        variants[f\'vendor@{mu}\'] = pop - mu * vendor\n        variants[f\'nplike@{mu}\'] = pop + mu * npz\n        variants[f\'classcons@{mu}\'] = pop + mu * cc\n        variants[f\'member@{mu}\'] = pop + mu * mem\n        variants[f\'stack@{mu}\'] = pop + mu * npz + mu * cc\n    for name, sc in variants.items():\n        rr, hit = expected_rank_metrics(sc, t)\n        out[f\'rr|{name}\'] = rr\n        for n, h in hit.items():\n            out[f\'hit{n}|{name}\'] = h\n    return out\n'
(WORK / "np_signals_core.py").write_text(NP_CORE_SRC)
sys.path.insert(0, str(WORK))

In [ ]:
# RDKit pinned to the scorer's version
print("python", PY_TAG, "| offline wheel:", RDKIT_WHL.name if RDKIT_WHL else "none for this Python -> PyPI")
cmd = ([sys.executable, "-m", "pip", "install", "-q", "--no-deps", str(RDKIT_WHL)] if RDKIT_WHL is not None
       else [sys.executable, "-m", "pip", "install", "-q", f"rdkit=={RDKIT_PIN}"])
r_ = subprocess.run(cmd, capture_output=True, text=True)
if r_.returncode != 0:
    raise RuntimeError("rdkit install failed:\n" + r_.stdout[-2000:] + "\n" + r_.stderr[-2000:])
import rdkit
assert rdkit.__version__ == RDKIT_PIN, rdkit.__version__
import np_signals_core as C
print("rdkit", rdkit.__version__, "| cpus", os.cpu_count())

## 1. Inputs (Drive) and PubChem downloads

In [ ]:
T0 = time.time()

def find(dst, name):
    hits = sorted(dst.rglob(name), key=lambda p: len(str(p)))
    assert hits, f"{name} not found under {dst}"
    return hits[0]

TIER_FILES = ["pc_mass.npy", "pc_off.npy", "pc_smiles.npy"]
POP_FILES = ["pc_cid.npy", "pc_ik14.npy", "pc_lsid.npy", "pc_lpmid.npy", "pool_cid.npy", "pool_lpmid.npy"]

def stage(src_dir, names, dst):
    """Copy the named .npy files from Drive to local disk (skipped if already there with the same size)."""
    dst.mkdir(parents=True, exist_ok=True)
    for n in names:
        s = find(src_dir, n)
        d = dst / n
        if not (d.exists() and d.stat().st_size == s.stat().st_size):
            t_ = time.time()
            subprocess.run(["cp", str(s), str(d)], check=True)
            print(f"  copied {n} {s.stat().st_size/1e9:.2f} GB in {time.time()-t_:.0f}s", flush=True)
    return dst

for d in (TIER_DRIVE, POP_DRIVE, NPT_DRIVE):
    assert d.exists(), f"missing on Drive: {d}"
TIER = stage(TIER_DRIVE, TIER_FILES, DL / "tier") if COPY_LOCAL else TIER_DRIVE
POP = stage(POP_DRIVE, POP_FILES, DL / "pop") if COPY_LOCAL else POP_DRIVE
NPT = NPT_DRIVE

def http_get(url, dst, tries=20, wait=30):
    if dst.exists() and dst.stat().st_size > 0:
        return dst
    for k in range(tries):
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0 (casmi26 np-signals build)"})
            with urllib.request.urlopen(req, timeout=600) as r, open(dst, "wb") as f:
                while True:
                    b = r.read(1 << 24)
                    if not b:
                        break
                    f.write(b)
            head = dst.read_bytes()[:200]
            if b"ServerBusy" in head or b"Status: 503" in head:
                raise IOError("PubChem busy")
            return dst
        except Exception as e:
            print(f"  {url} try {k+1}: {e!r}; retry in {wait}s", flush=True)
            dst.unlink(missing_ok=True)
            time.sleep(wait)
    raise RuntimeError(f"could not fetch {url}")

SIDMAP = http_get(FTP + "SID-Map.gz", DL / "SID-Map.gz")
SRCNAMES = http_get(FTP + "Source-Names", DL / "Source-Names")
SRCTAB = http_get(SRCTAB_URL, DL / "sourcetable_substance.csv")
print(f"downloads done {time.time()-T0:.0f}s | SID-Map {SIDMAP.stat().st_size/1e9:.2f} GB")

mass = np.load(find(TIER, "pc_mass.npy"), mmap_mode="r")
off = np.load(find(TIER, "pc_off.npy"), mmap_mode="r")
buf = np.load(find(TIER, "pc_smiles.npy"), mmap_mode="r")
pc_cid = np.load(find(POP, "pc_cid.npy"), mmap_mode="r")
pc_ik14 = np.load(find(POP, "pc_ik14.npy"), mmap_mode="r")
pc_lsid = np.load(find(POP, "pc_lsid.npy"), mmap_mode="r")
pc_lpmid = np.load(find(POP, "pc_lpmid.npy"), mmap_mode="r")
pool_cid = np.load(find(POP, "pool_cid.npy"), mmap_mode="r")
pool_lpmid = np.load(find(POP, "pool_lpmid.npy"), mmap_mode="r")
N_TIER = len(mass)
assert N_TIER == 105_875_489 and len(pc_cid) == N_TIER and len(pc_ik14) == N_TIER, (N_TIER, len(pc_cid))
assert len(pool_cid) == 710_701, len(pool_cid)
npt = pd.read_parquet(find(NPT, "natural_products.parquet"))
print("tier", N_TIER, "| pool", len(pool_cid), "| NP table", len(npt))

## 2. `vendor_only`

In [ ]:
T1 = time.time()
srctab = pd.read_csv(SRCTAB, dtype=str)
print("source table columns:", list(srctab.columns))
srcnames = pd.read_csv(SRCNAMES, sep="\t", header=None, dtype=str, quoting=3, on_bad_lines="skip")
try:
    VEND = C.vendor_source_ids(srctab, srcnames)
except KeyError as e:
    # no category column: the vendor signal is VOID, but NP-likeness and class consistency still get benchmarked
    print("VENDOR VOID:", e); VEND = set()
print("vendor source names/ids:", len(VEND))

MAXCID = int(max(np.max(pc_cid), np.max(pool_cid), pd.to_numeric(npt.pubchem_cid, errors="coerce").max())) + 1
nonv = np.zeros(MAXCID, bool)
anys = np.zeros(MAXCID, bool)
src_sids = {}
n_sid = 0
for ch in pd.read_csv(SIDMAP, sep="\t", header=None, names=["sid", "src", "reg", "cid"], usecols=["src", "cid"],
                      dtype={"src": str}, quoting=3, chunksize=20_000_000, on_bad_lines="skip"):
    n_sid += C.accumulate_sid_map(ch.src.values, pd.to_numeric(ch.cid, errors="coerce").values, VEND, nonv, anys)
    for s, n in ch.src.value_counts().items():
        src_sids[s] = src_sids.get(s, 0) + int(n)
    print(f"  SID-Map {n_sid/1e6:.0f}M SIDs with a CID, {time.time()-T1:.0f}s", flush=True)

top = sorted(src_sids.items(), key=lambda kv: -kv[1])[:40]
print("\nTop-40 depositors by SIDs (V = counted as vendor). Check that big screening aggregators are V:")
for s, n in top:
    print(f"  {'V' if s.strip() in VEND else ' '} {n:>12,}  {s}")

pc_vendor = np.zeros(N_TIER, np.uint8)
for i in range(0, N_TIER, 10_000_000):
    j = min(N_TIER, i + 10_000_000)
    pc_vendor[i:j] = C.vendor_only_flags(np.asarray(pc_cid[i:j]), np.asarray(pc_lpmid[i:j]), nonv, anys)
pool_vendor = C.vendor_only_flags(np.asarray(pool_cid), np.asarray(pool_lpmid), nonv, anys)
np.save(OUT / "pc_vendor_only.npy", pc_vendor)
np.save(OUT / "pool_vendor_only.npy", pool_vendor)

ncid = pd.to_numeric(npt.pubchem_cid, errors="coerce").fillna(0).astype(np.int64).values
np_vendor = C.vendor_only_flags(ncid, np.log1p(npt.pubmed_articles.fillna(0).values), nonv, anys)
V0 = float(np_vendor[ncid > 0].mean())
print(f"\nvendor_only rate: tier {pc_vendor.mean():.3f} | pool {pool_vendor.mean():.3f} | "
      f"NP table (NB.V0) {V0:.4f} (gate < {GATE_V0})")
NBV0 = bool(VEND) and V0 < GATE_V0
print(f"NB.V0 {'PASS' if NBV0 else 'FAIL'} | step 2 {time.time()-T1:.0f}s")

## 3. Labels: NP table, LOTUS kNN bank, training structures

In [ ]:
from rdkit import Chem, RDLogger
from rdkit.Chem.Descriptors import ExactMolWt
from rdkit.Chem.rdMolDescriptors import CalcMolFormula
RDLogger.DisableLog("rdApp.*")
T2 = time.time()

lab = npt[npt.np_superclass.notna()].drop_duplicates("ik14")
LABEL_OF = dict(zip(lab.ik14, lab.np_superclass))

def _fp_bits(s):
    fp = C.morgan_fp(s)
    if fp is None:
        return None
    a = np.zeros(2048, np.uint8)
    from rdkit import DataStructs
    DataStructs.ConvertToNumpyArray(fp, a)
    return np.packbits(a)

with mp.get_context("fork").Pool(os.cpu_count()) as P:
    bits = P.map(_fp_bits, lab.smiles.tolist(), chunksize=512)
okb = np.array([b is not None for b in bits])
lotus_fp = np.stack([b for b in bits if b is not None])
lotus_lab = lab.np_superclass.values[okb].astype(str)
np.save(OUT / "lotus_fp.npy", lotus_fp)
np.save(OUT / "lotus_superclass.npy", lotus_lab)
np.save(OUT / "lotus_ik14.npy", lab.ik14.values[okb].astype("S14"))
print(f"LOTUS bank {len(lotus_fp)} labelled structures, {len(set(lotus_lab))} superclasses")

def _prep_train(s):
    m = Chem.MolFromSmiles(s)
    if m is None:
        return None
    return Chem.MolToInchiKey(m)[:14], CalcMolFormula(m), ExactMolWt(m)

tr = pd.read_parquet(TRAIN, columns=["inchikey14", "normalized_smiles"]).drop_duplicates("inchikey14")
with mp.get_context("fork").Pool(os.cpu_count()) as P:
    pr = P.map(_prep_train, tr.normalized_smiles.tolist(), chunksize=256)
tr = tr.assign(ik=[p[0] if p else None for p in pr], formula=[p[1] if p else None for p in pr],
               mono=[p[2] if p else np.nan for p in pr]).dropna(subset=["ik"]).drop_duplicates("ik").reset_index(drop=True)

def _bv(packed):
    from rdkit import DataStructs
    return DataStructs.CreateFromBitString("".join(map(str, np.unpackbits(packed))))

LOTUS_FPS = [_bv(b) for b in lotus_fp]
C.G.update(lotus_fps=LOTUS_FPS, lotus_labels=list(lotus_lab))

def _train_label(i):
    k = tr.ik.iat[i]
    if k in LABEL_OF:
        return LABEL_OF[k], "table"
    l, _ = C.knn_label(C.morgan_fp(tr.normalized_smiles.iat[i]), C.G["lotus_fps"], C.G["lotus_labels"], C.KNN_THR)
    return l, ("knn" if l is not None else "none")

with mp.get_context("fork").Pool(os.cpu_count()) as P:
    tl = P.map(_train_label, range(len(tr)), chunksize=64)
tr["superclass"] = [x[0] for x in tl]
tr["label_source"] = [x[1] for x in tl]
tr[["ik", "inchikey14", "normalized_smiles", "formula", "mono", "superclass", "label_source"]].to_parquet(
    OUT / "train_labels.parquet", index=False)
print("training structures", len(tr), "| label source:", tr.label_source.value_counts().to_dict(),
      f"| {time.time()-T2:.0f}s")

## 4. Cohorts and 5. the benchmark

In [ ]:
T3 = time.time()
rng = np.random.default_rng(SEED)
cT = tr[(tr.mono >= 150) & (tr.mono <= 1250)]
cT = cT.iloc[rng.permutation(len(cT))[:N_COHORT]]
cL = npt[(~npt.in_casmi_train.astype(bool)) & npt.mono_mass.between(150, 1250)].drop_duplicates("ik14")
cL = cL.iloc[rng.permutation(len(cL))[:N_COHORT]]
tasks = [(f"T{i}", r.ik, r.formula, r.mono, r.normalized_smiles, "T") for i, r in enumerate(cT.itertuples())]
tasks += [(f"L{i}", r.ik14, r.formula, r.mono_mass, r.smiles, "L") for i, r in enumerate(cL.itertuples())]

C.G.update(mass=mass, off=off, buf=buf, ik14=pc_ik14, lsid=pc_lsid, lpmid=pc_lpmid, vendor=pc_vendor, ppm=PPM_WIN,
           label_of={**LABEL_OF, **{k: v for k, v in zip(tr.ik, tr.superclass) if v is not None}},
           train_fps=[C.morgan_fp(s) for s in tr.normalized_smiles], train_labels=list(tr.superclass),
           train_rows_of={k: [i] for i, k in enumerate(tr.ik)})

rows = []
with mp.get_context("fork").Pool(os.cpu_count()) as P:
    for k, r in enumerate(P.imap_unordered(C.eval_query, tasks, chunksize=4)):
        rows.append(r)
        if (k + 1) % 250 == 0:
            print(f"  {k+1}/{len(tasks)} {time.time()-T3:.0f}s", flush=True)
B = pd.DataFrame(rows)
B.to_parquet(OUT / "np_bench_rows.parquet", index=False)

In [ ]:
RES = dict(rdkit=rdkit.__version__, n_tier=N_TIER, ppm=PPM_WIN, seed=SEED, vendor_sources=len(VEND),
           vendor_rate_tier=float(pc_vendor.mean()), vendor_rate_pool=float(pool_vendor.mean()), NB_V0=V0)
print("\n=== Track 4b offline benchmark ===")
for co in ("T", "L"):
    b = B[B.cohort == co]
    RES[f"found_{co}"] = float(b.found.mean())
    print(f"cohort {co}: {len(b)} sampled, truth in its same-formula tier window {b.found.mean():.1%}, "
          f"median window {int(b[b.found].n.median())}")
F = B[B.found].copy()
assert len(F), "no truth found in the tier"
names = sorted({c.split("|", 1)[1] for c in F.columns if c.startswith("rr|")}, key=lambda s: (s != "pop", s))
tab = []
for nm in names:
    row = dict(variant=nm)
    for co in ("T", "L"):
        f = F[F.cohort == co]
        row[f"MRR_{co}"] = f[f"rr|{nm}"].mean()
        row[f"hit1_{co}"] = f[f"hit1|{nm}"].mean()
        row[f"hit25_{co}"] = f[f"hit25|{nm}"].mean()
    tab.append(row)
TAB = pd.DataFrame(tab)
for co in ("T", "L"):
    TAB[f"dMRR_{co}"] = TAB[f"MRR_{co}"] - TAB.loc[TAB.variant == "pop", f"MRR_{co}"].iloc[0]
pd.set_option("display.width", 200)
print(TAB.round(4).to_string(index=False))

def gate(sig):
    t = TAB[TAB.variant.str.startswith(sig + "@")]
    best = t.loc[t.dMRR_L.idxmax()]
    ok = bool(best.dMRR_L >= GATE_L and best.dMRR_T >= GATE_T)
    return dict(best_mu=float(best.variant.split("@")[1]), dMRR_L=float(best.dMRR_L), dMRR_T=float(best.dMRR_T), pass_=ok)

RES["NB_V"] = gate("vendor")
RES["NB_N"] = gate("nplike")
c0a = float(F.has_lab_analog.mean())
c0b = float(F.multi_class_window.mean())
RES["NB_C0"] = dict(lab_analog=c0a, multi_class=c0b, pass_=bool(c0a >= GATE_C0_ANALOG and c0b >= GATE_C0_MULTI))
RES["NB_C"] = gate("classcons")
RES["truth_vendor_rate"] = {co: float(F[F.cohort == co].truth_vendor.mean()) for co in ("T", "L")}
RES["truth_labelled"] = {co: float(F[F.cohort == co].truth_lab.mean()) for co in ("T", "L")}
RES["table"] = TAB.round(5).to_dict(orient="records")
print(f"\nNB.V0 vendor_only on NP-table compounds {V0:.4f} -> {'PASS' if NBV0 else 'FAIL'}")
for g in ("NB_V", "NB_N", "NB_C"):
    r = RES[g]
    print(f"{g}: best mu {r['best_mu']} | dMRR L {r['dMRR_L']:+.4f} (>= {GATE_L}) | T {r['dMRR_T']:+.4f} (>= {GATE_T}) "
          f"-> {'PASS' if r['pass_'] else 'FAIL'}")
print(f"NB_C0: labelled analog {c0a:.1%} (>= {GATE_C0_ANALOG:.0%}) | windows with >= 2 superclasses {c0b:.1%} "
      f"(>= {GATE_C0_MULTI:.0%}) -> {'PASS' if RES['NB_C0']['pass_'] else 'FAIL'}")
print("truth vendor_only rate:", RES["truth_vendor_rate"], "| truth labelled:", RES["truth_labelled"])
print("Caveat: cohort L truths always carry an NP-table label, so classcons on L is optimistic; read it with T.")
if not NBV0:
    print("NB.V0 FAILED: the vendor categorisation is wrong; NB.V is VOID whatever it says.")

MANIFEST = dict(built=time.strftime("%Y-%m-%d"), rdkit=rdkit.__version__, tier_rows=N_TIER, pool_rows=len(pool_cid),
                tier_src=str(TIER_DRIVE), pop_src=str(POP_DRIVE), npt_src=str(NPT_DRIVE), sidmap_bytes=SIDMAP.stat().st_size,
                files=sorted(p.name for p in OUT.iterdir()))
(OUT / "MANIFEST.json").write_text(json.dumps(MANIFEST, indent=1))
(OUT / "np_bench.json").write_text(json.dumps(RES, indent=1, default=float))
print(f"\nwrote {OUT} | total {time.time()-T0:.0f}s")